In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import os
import numpy as np
import torch.optim as optim
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from tqdm import tqdm

In [ ]:
class DoubleConv(nn.Module):
    """(convolution => [BN] => ReLU) * 2"""
    def __init__(self, in_channels, out_channels, mid_channels=None):
        super().__init__()
        if not mid_channels:
            mid_channels = out_channels
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.double_conv(x)
class Down(nn.Module):
    """Downscaling with maxpool then double conv"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )
    def forward(self, x):
        return self.maxpool_conv(x)
class AttentionBlock(nn.Module):
    """
    Attention Gate (AG) Module.
    Calculates spatial attention coefficients alpha in [0, 1] to weight encoder skip features.
    """
    def __init__(self, F_g, F_l, F_int):
        """
        F_g: Number of feature maps in gating signal (from lower/decoder layer)
        F_l: Number of feature maps in skip connection (from encoder layer)
        F_int: Intermediate channel capacity
        """
        super(AttentionBlock, self).__init__()
        self.W_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(F_int)
        )
        
        self.W_x = nn.Sequential(
            nn.Conv2d(F_l, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(F_int)
        )
        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )
        
        self.relu = nn.ReLU(inplace=True)
    def forward(self, g, x):
        # g: (B, F_g, H_g, W_g)
        # x: (B, F_l, H_x, W_x)
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        
        # If spatial dimensions differ, upsample g1 to match x1
        if g1.size()[2:] != x1.size()[2:]:
            g1 = F.interpolate(g1, size=x1.size()[2:], mode='bilinear', align_corners=True)
            
        relu = self.relu(g1 + x1)
        alpha = self.psi(relu)
        
        # Multiply encoder feature map by attention weights
        return x * alpha
class UpAttention(nn.Module):
    """Upscaling, applying Attention Gate, then double conv"""
    def __init__(self, in_channels, out_channels, bilinear=True):
        super().__init__()
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)
            
        # Attention Gate for skip connection
        # Encoder channels (F_l) = in_channels // 2, Gating channels (F_g) = in_channels // 2
        self.ag = AttentionBlock(F_g=in_channels // 2, F_l=in_channels // 2, F_int=in_channels // 4)
    def forward(self, x1, x2):
        # x1: Decoder gating signal (smaller spatial, larger channels)
        # x2: Encoder skip features (larger spatial)
        x1 = self.up(x1)
        
        # Handle padding differences if any
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]
        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2, diffY // 2, diffY - diffY // 2])
        
        # Apply Attention Gate to encoder skip features
        x2_attended = self.ag(g=x1, x=x2)
        
        # Concatenate attended features with upsampled features
        x = torch.cat([x2_attended, x1], dim=1)
        return self.conv(x)
class OutConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)
    def forward(self, x):
        return self.conv(x)
class AttentionUNet(nn.Module):
    """
    Attention U-Net Architecture for Medical Image Segmentation.
    Integrates spatial Attention Gates at all skip connections.
    """
    def __init__(self, n_channels=1, n_classes=4, bilinear=False):
        super(AttentionUNet, self).__init__()
        self.n_channels = n_channels
        self.n_classes = n_classes
        self.bilinear = bilinear
        # Encoder Path
        self.inc = DoubleConv(n_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        factor = 2 if bilinear else 1
        self.down4 = Down(512, 1024 // factor)
        # Decoder Path with Attention Gates
        self.up1 = UpAttention(1024, 512 // factor, bilinear)
        self.up2 = UpAttention(512, 256 // factor, bilinear)
        self.up3 = UpAttention(256, 128 // factor, bilinear)
        self.up4 = UpAttention(128, 64, bilinear)
        
        self.outc = OutConv(64, n_classes)
    def forward(self, x):
        # Encoder
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        
        # Decoder with Attention Filtering
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        logits = self.outc(x)
        return logits

In [ ]:
class DiceLoss(nn.Module):
    def __init__(self, n_classes=4, smooth=1e-6):
        super().__init__()
        self.n_classes = n_classes
        self.smooth = smooth
    def forward(self, logits, targets):
        probs = torch.softmax(logits, dim=1)
        targets_one_hot = torch.eye(self.n_classes, device=logits.device)[targets]
        targets_one_hot = targets_one_hot.permute(0, 3, 1, 2).float()
        
        dice_loss = 0.0
        for class_idx in range(1, self.n_classes):
            p = probs[:, class_idx, ...]
            t = targets_one_hot[:, class_idx, ...]
            
            intersection = torch.sum(p * t)
            cardinality = torch.sum(p + t)
            
            class_dice = (2. * intersection + self.smooth) / (cardinality + self.smooth)
            dice_loss += (1.0 - class_dice)
            
        return dice_loss / (self.n_classes - 1)
def compute_dice_coefficient(preds, targets, class_idx):
    pred_mask = (preds == class_idx)
    target_mask = (targets == class_idx)
    
    intersection = torch.sum(pred_mask & target_mask).float()
    cardinality = (torch.sum(pred_mask) + torch.sum(target_mask)).float()
    
    if cardinality == 0:
        return 1.0
    return (2. * intersection / cardinality).item()
def train_one_epoch(model, loader, criterion_ce, criterion_dice, optimizer, device):
    model.train()
    epoch_loss = 0.0
    
    progress_bar = tqdm(loader, desc="  Training Attention U-Net", leave=False)
    for images, masks in progress_bar:
        images = images.to(device)
        masks = masks.to(device)
        
        optimizer.zero_grad()
        logits = model(images)
        
        loss_ce = criterion_ce(logits, masks)
        loss_dice = criterion_dice(logits, masks)
        loss = loss_ce + loss_dice
        
        loss.backward()
        optimizer.step()
        
        loss_val = loss.item()
        epoch_loss += loss_val * images.size(0)
        progress_bar.set_postfix(loss=f"{loss_val:.4f}")
        
    return epoch_loss / len(loader.dataset)
@torch.no_grad()
def validate(model, loader, criterion_ce, criterion_dice, device):
    model.eval()
    epoch_loss = 0.0
    dice_scores = {1: [], 2: [], 3: []}
    
    progress_bar = tqdm(loader, desc="  Validating", leave=False)
    for images, masks in progress_bar:
        images = images.to(device)
        masks = masks.to(device)
        
        logits = model(images)
        loss_ce = criterion_ce(logits, masks)
        loss_dice = criterion_dice(logits, masks)
        loss = loss_ce + loss_dice
        
        epoch_loss += loss.item() * images.size(0)
        preds = torch.argmax(logits, dim=1)
        
        for class_idx in [1, 2, 3]:
            dsc = compute_dice_coefficient(preds, masks, class_idx)
            dice_scores[class_idx].append(dsc)
            
    avg_loss = epoch_loss / len(loader.dataset)
    mean_dice = {k: np.mean(v) for k, v in dice_scores.items()}
    avg_dice = np.mean(list(mean_dice.values()))
    
    return avg_loss, mean_dice, avg_dice
def main():
    preproc_dir = r"C:\Users\NITRO V 15\.gemini\antigravity\scratch\preprocessed_data_numpy"
    
    if not os.path.exists(preproc_dir):
        print(f"[Error] Preprocessed directory '{preproc_dir}' does not exist.")
        return
        
    BATCH_SIZE = 16
    LEARNING_RATE = 2e-4
    EPOCHS = 25
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Running Attention U-Net Training ({EPOCHS} Epochs + Class-Weighted Loss) on Device: {DEVICE}")
    
    all_train_pids = [f"patient{str(i).zfill(3)}" for i in range(1, 101)]
    train_pids = all_train_pids[:80]
    val_pids = all_train_pids[80:]
    
    print("\nInitializing Data Loaders...")
    train_dataset = ACDCDataset(preproc_dir, split='train', patient_ids=train_pids, augment=True)
    val_dataset = ACDCDataset(preproc_dir, split='train', patient_ids=val_pids, augment=False)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    
    # Initialize Attention U-Net
    model = AttentionUNet(n_channels=1, n_classes=4, bilinear=False).to(DEVICE)
    
    # Class-Weighted Cross-Entropy Loss: Background=0.2, RV=1.5, MYO=2.0, LV=1.0
    class_weights = torch.tensor([0.2, 1.5, 2.0, 1.0], device=DEVICE)
    criterion_ce = nn.CrossEntropyLoss(weight=class_weights)
    criterion_dice = DiceLoss(n_classes=4)
    
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
    
    history = {
        "train_loss": [], "val_loss": [],
        "val_dice_avg": [], "val_dice_rv": [], "val_dice_myo": [], "val_dice_lv": [],
        "lr": []
    }
    
    best_dice = 0.0
    print("\nStarting Training Loop...")
    for epoch in range(1, EPOCHS + 1):
        current_lr = optimizer.param_groups[0]['lr']
        train_loss = train_one_epoch(model, train_loader, criterion_ce, criterion_dice, optimizer, DEVICE)
        val_loss, val_dice_dict, val_dice_avg = validate(model, val_loader, criterion_ce, criterion_dice, DEVICE)
        
        scheduler.step()
        
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_dice_avg"].append(val_dice_avg)
        history["val_dice_rv"].append(val_dice_dict[1])
        history["val_dice_myo"].append(val_dice_dict[2])
        history["val_dice_lv"].append(val_dice_dict[3])
        history["lr"].append(current_lr)
        
        print(f"Epoch {epoch:02d}/{EPOCHS:02d} (LR={current_lr:.6f}): "
              f"Train Loss={train_loss:.4f} | Val Loss={val_loss:.4f} | "
              f"Val Dice={val_dice_avg:.4f} (RV={val_dice_dict[1]:.3f}, MYO={val_dice_dict[2]:.3f}, LV={val_dice_dict[3]:.3f})")
              
        if val_dice_avg > best_dice:
            best_dice = val_dice_avg
            torch.save(model.state_dict(), "best_attention_unet_model.pth")
            print("  --> Saved best Attention U-Net checkpoint (best_attention_unet_model.pth).")
            
    plt.figure(figsize=(15, 5))
    
    plt.subplot(1, 3, 1)
    plt.plot(history["train_loss"], label="Train Loss")
    plt.plot(history["val_loss"], label="Val Loss")
    plt.title("Attention U-Net Loss Progression")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    
    plt.subplot(1, 3, 2)
    plt.plot(history["val_dice_avg"], label="Average Dice", color='black', linewidth=2)
    plt.plot(history["val_dice_rv"], label="RV (Class 1)", linestyle='--')
    plt.plot(history["val_dice_myo"], label="MYO (Class 2)", linestyle='--')
    plt.plot(history["val_dice_lv"], label="LV (Class 3)", linestyle='--')
    plt.title("Validation Dice Coefficients")
    plt.xlabel("Epoch")
    plt.ylabel("Dice Score")
    plt.legend()
    
    plt.subplot(1, 3, 3)
    plt.plot(history["lr"], color='purple')
    plt.title("Learning Rate Decay")
    plt.xlabel("Epoch")
    plt.ylabel("LR")
    
    plt.tight_layout()
    plt.savefig("attention_unet_training_curves.png", dpi=150)
    plt.close()
    
    print("\nAttention U-Net Training Complete! Weights saved to 'best_attention_unet_model.pth'.")
if __name__ == "__main__":
    main()